# 从零复现 NeRF：相机射线、位置编码、体渲染与 coarse/fine 分层采样

本 Notebook 不导入任何现成 NeRF、渲染器或预训练模型。我们用基础 PyTorch 手写 pinhole camera rays、sin/cos positional encoding、带 skip connection 的 `NeRFMLP.forward`、分层采样、alpha compositing、体渲染，以及 coarse/fine hierarchical renderer。

数值 oracle 覆盖常密度解析解、权重和与背景颜色、near/far 合同、显式 generator 可复现性、shape/finite/梯度；最后拟合一个解析微型辐射场，并只在声明的视角包络内做 novel-view 检查。全部离线、CPU 单线程；这是算法复现与工程边界测试，不是高保真新视角合成结果。


## 1. 从像素到颜色的计算图

```text
pixel centers + intrinsics + camera-to-world
  -> ray origin o [R,3], unit direction d [R,3]
  -> sample depths z [R,S]
  -> points x=o+zd [R,S,3]
  -> positional encodings -> NeRF MLP -> sigma [R,S], rgb [R,S,3]
  -> alpha compositing -> coarse rgb/depth/weights
  -> PDF(weights) -> fine depths -> merge/sort -> fine rendering
```

射线颜色的连续形式是

$$C(r)=\int_{t_n}^{t_f}T(t)\sigma(r(t))c(r(t),d)\,dt,\qquad
T(t)=\exp\left(-\int_{t_n}^{t}\sigma(r(s))ds\right).$$

离散实现最危险的不是 MLP，而是坐标约定、区间长度、排序、最后背景透射率和随机流。


In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)

from copy import deepcopy
from dataclasses import dataclass
from hashlib import sha256
from types import MappingProxyType
import io
import json
import math
import random
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED = 480728
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.set_num_threads(1)
DEVICE = torch.device("cpu")

assert DEVICE.type == "cpu"
assert torch.get_num_threads() == 1
print({"torch": torch.__version__, "device": str(DEVICE), "seed": SEED})


## 2. Pinhole rays：刚体结构校验不等于 pose 方向认证

本册约定 camera space 的相机看向 `-z`；像素索引 `y` 向下，而 camera `+y` 向上，因此像素中心方向为

$$d_c=((x+0.5-W/2)/f,;-(y+0.5-H/2)/f,;-1).$$

正交旋转、行列式 1 和齐次底行只能证明矩阵是**合法刚体变换**。一个 C2W 的逆矩阵同样满足这些性质，所以结构检查绝不声称能单独区分 C2W/W2C。

工程上使用 `CameraToWorld` 受信类型：发布方 registry 绑定 `camera_id -> canonical matrix digest`，ray generator 每次读取时重新核验，防止调用方把结构合法但语义相反的矩阵冒充已发布相机。已知平移相机的 origin/中心 ray 构成坐标 oracle。


In [ ]:
def validate_rigid_transform48(matrix):
    if matrix.ndim != 3 or matrix.shape[-2:] != (4, 4):
        raise ValueError("camera matrix must be floating [B,4,4]")
    if not torch.is_floating_point(matrix) or not torch.isfinite(matrix).all():
        raise ValueError("camera matrices must be finite floating point")
    rotation = matrix[:, :3, :3]
    identity = torch.eye(3, device=rotation.device, dtype=rotation.dtype).expand(rotation.shape[0], -1, -1)
    if not torch.allclose(rotation.transpose(1, 2) @ rotation, identity, atol=1e-4, rtol=1e-4):
        raise ValueError("camera rotation must be orthonormal")
    if not torch.allclose(torch.linalg.det(rotation), torch.ones(rotation.shape[0], device=rotation.device,
                                                                  dtype=rotation.dtype), atol=1e-4):
        raise ValueError("camera rotation must be proper")
    expected_bottom = torch.tensor([0., 0., 0., 1.], device=matrix.device, dtype=matrix.dtype)
    if not torch.allclose(matrix[:, 3], expected_bottom.expand(matrix.shape[0], -1)):
        raise ValueError("invalid homogeneous transform bottom row")

def camera_matrix_digest48(matrix):
    value = matrix.detach().cpu().contiguous()
    digest = sha256()
    digest.update(str(value.dtype).encode())
    digest.update(json.dumps(list(value.shape)).encode())
    digest.update(value.numpy().tobytes())
    return digest.hexdigest()

def look_at_pose48(angle_degrees, radius=2.0):
    if not math.isfinite(angle_degrees) or not math.isfinite(radius) or radius <= 0:
        raise ValueError("look-at angle/radius contract is invalid")
    angle = math.radians(angle_degrees)
    origin = torch.tensor([radius * math.sin(angle), 0.0, radius * math.cos(angle)])
    forward = F.normalize(-origin, dim=0)
    world_up = torch.tensor([0., 1., 0.])
    right = F.normalize(torch.cross(forward, world_up, dim=0), dim=0)
    up = torch.cross(right, forward, dim=0)
    pose = torch.eye(4)
    pose[:3, :3] = torch.stack([right, up, -forward], dim=1)
    pose[:3, 3] = origin
    return pose

IDENTITY_POSE48 = torch.eye(4).unsqueeze(0)
TRANSLATED_POSE48 = torch.eye(4).unsqueeze(0)
TRANSLATED_POSE48[0, :3, 3] = torch.tensor([0., 0., 2.])
EVALUATION_POSE48 = look_at_pose48(15.0, 2.0).unsqueeze(0)
_CAMERA_REGISTRY48 = MappingProxyType({
    "identity-oracle": camera_matrix_digest48(IDENTITY_POSE48),
    "translated-z2-oracle": camera_matrix_digest48(TRANSLATED_POSE48),
    "eval-azimuth-15-radius-2": camera_matrix_digest48(EVALUATION_POSE48),
})

@dataclass(frozen=True)
class CameraToWorld:
    camera_id: str
    matrix: torch.Tensor
    matrix_digest: str

def trusted_camera_to_world48(camera_id, matrix):
    if camera_id not in _CAMERA_REGISTRY48:
        raise ValueError("camera_id is not approved")
    validate_rigid_transform48(matrix)
    actual = camera_matrix_digest48(matrix)
    if actual != _CAMERA_REGISTRY48[camera_id]:
        raise ValueError("camera matrix digest does not match trusted camera_id")
    return CameraToWorld(camera_id, matrix.detach().clone(), actual)

class PinholeRayGenerator(nn.Module):
    def __init__(self, height, width, focal):
        super().__init__()
        if min(height, width) <= 0 or not math.isfinite(focal) or focal <= 0:
            raise ValueError("image dimensions and focal must be positive")
        self.height, self.width, self.focal = int(height), int(width), float(focal)
        yy, xx = torch.meshgrid(torch.arange(height, dtype=torch.float32),
                                torch.arange(width, dtype=torch.float32), indexing="ij")
        directions = torch.stack([(xx + 0.5 - width / 2) / focal,
                                  -(yy + 0.5 - height / 2) / focal,
                                  -torch.ones_like(xx)], dim=-1)
        self.register_buffer("camera_directions", directions)

    def forward(self, camera):
        if not isinstance(camera, CameraToWorld):
            raise ValueError("ray generation requires a trusted CameraToWorld")
        if camera.camera_id not in _CAMERA_REGISTRY48:
            raise ValueError("camera_id is no longer trusted")
        matrix = camera.matrix
        validate_rigid_transform48(matrix)
        actual = camera_matrix_digest48(matrix)
        if actual != camera.matrix_digest or actual != _CAMERA_REGISTRY48[camera.camera_id]:
            raise ValueError("trusted camera digest changed")
        if matrix.dtype != self.camera_directions.dtype or matrix.device != self.camera_directions.device:
            raise ValueError("camera matrix must match ray generator dtype/device")
        rotation = matrix[:, :3, :3]
        camera_dirs = self.camera_directions.reshape(1, -1, 3).expand(rotation.shape[0], -1, -1)
        world_dirs = camera_dirs @ rotation.transpose(1, 2)
        world_dirs = F.normalize(world_dirs, dim=-1)
        origins = matrix[:, None, :3, 3].expand_as(world_dirs)
        return origins.reshape(-1, 3), world_dirs.reshape(-1, 3)

identity_pose = IDENTITY_POSE48
identity_camera48 = trusted_camera_to_world48("identity-oracle", identity_pose)
ray_oracle = PinholeRayGenerator(5, 5, 5.0)
origins_oracle, directions_oracle = ray_oracle(identity_camera48)
assert origins_oracle.shape == directions_oracle.shape == (25, 3)
assert torch.equal(origins_oracle, torch.zeros_like(origins_oracle))
assert torch.allclose(directions_oracle[12], torch.tensor([0., 0., -1.]))
assert torch.allclose(directions_oracle.norm(dim=-1), torch.ones(25))

translated_camera48 = trusted_camera_to_world48("translated-z2-oracle", TRANSLATED_POSE48)
translated_origins48, translated_directions48 = ray_oracle(translated_camera48)
assert torch.equal(translated_origins48[12], torch.tensor([0., 0., 2.]))
assert torch.allclose(translated_directions48[12], torch.tensor([0., 0., -1.]))

# 逆矩阵本身仍是合法刚体，但不能冒充相同 camera_id。
inverse_pose48 = torch.linalg.inv(TRANSLATED_POSE48)
validate_rigid_transform48(inverse_pose48)
try:
    trusted_camera_to_world48("translated-z2-oracle", inverse_pose48)
    raise AssertionError("W2C inverse must not impersonate a registered C2W")
except ValueError as exc:
    assert "digest" in str(exc)

bad_pose = identity_pose.clone(); bad_pose[0, 0, 0] = 2
try:
    trusted_camera_to_world48("identity-oracle", bad_pose)
    raise AssertionError("non-orthonormal camera must fail")
except ValueError:
    pass

try:
    ray_oracle(identity_pose)
    raise AssertionError("raw matrix must not bypass CameraToWorld trust")
except ValueError as exc:
    assert "CameraToWorld" in str(exc)


## 3. Positional encoding：让 MLP 表达高频空间变化

普通 MLP 对高频细节存在 spectral bias。NeRF 把每个标量映射到

$$\gamma(p)=[p,\sin(2^0\pi p),\cos(2^0\pi p),\ldots,\sin(2^{L-1}\pi p),\cos(2^{L-1}\pi p)].$$

输出维度为 `input_dim*(1+2L)`。频率、是否乘 $\pi$、是否包含原输入都属于权重语义的一部分，必须进入 artifact。


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, input_dim=3, num_frequencies=5, include_input=True):
        super().__init__()
        if input_dim <= 0 or num_frequencies <= 0:
            raise ValueError("encoding dimensions must be positive")
        self.input_dim, self.num_frequencies = int(input_dim), int(num_frequencies)
        self.include_input = bool(include_input)
        self.output_dim = input_dim * (2 * num_frequencies + int(include_input))
        self.register_buffer("frequencies", (2.0 ** torch.arange(num_frequencies)) * math.pi)

    def forward(self, x):
        if x.shape[-1] != self.input_dim or not torch.is_floating_point(x) or not torch.isfinite(x).all():
            raise ValueError("finite floating input with configured last dimension required")
        angles = x.unsqueeze(-1) * self.frequencies
        encoded = [x] if self.include_input else []
        encoded.extend([angles.sin().flatten(-2), angles.cos().flatten(-2)])
        return torch.cat(encoded, dim=-1)

xyz_encoding48 = PositionalEncoding(3, 5, True)
dir_encoding48 = PositionalEncoding(3, 2, True)
pe_zero = xyz_encoding48(torch.zeros(4, 3))
assert pe_zero.shape == (4, 33)
assert torch.equal(pe_zero[:, :3], torch.zeros(4, 3))
assert torch.equal(pe_zero[:, 3:18], torch.zeros(4, 15))
assert torch.equal(pe_zero[:, 18:], torch.ones(4, 15))


## 4. 手写 NeRF MLP：density 只看位置，颜色还看方向

位置编码经过多层 MLP，在中间层拼回原始位置编码形成 skip connection；该 concat 应发生在对应 linear **之前**。density 用 `softplus` 保证非负。颜色分支把几何 feature 与方向编码拼接，再经 sigmoid 落到 `[0,1]`。

这种分支结构表达“同一点的密度不随观察方向改变，但表面颜色可以有 view-dependent effect”。


In [ ]:
class NeRFMLP(nn.Module):
    def __init__(self, xyz_frequencies=5, dir_frequencies=2, hidden_dim=48, depth=5, skip_layer=3):
        super().__init__()
        if depth < 2 or not 0 < skip_layer < depth or hidden_dim <= 0:
            raise ValueError("invalid NeRF MLP structure")
        self.xyz_encoding = PositionalEncoding(3, xyz_frequencies, True)
        self.dir_encoding = PositionalEncoding(3, dir_frequencies, True)
        self.hidden_dim, self.depth, self.skip_layer = int(hidden_dim), int(depth), int(skip_layer)
        layers = []
        for index in range(depth):
            if index == 0:
                input_dim = self.xyz_encoding.output_dim
            elif index == skip_layer:
                input_dim = hidden_dim + self.xyz_encoding.output_dim
            else:
                input_dim = hidden_dim
            layers.append(nn.Linear(input_dim, hidden_dim))
        self.position_layers = nn.ModuleList(layers)
        self.sigma_head = nn.Linear(hidden_dim, 1)
        self.feature_head = nn.Linear(hidden_dim, hidden_dim)
        self.direction_layer = nn.Linear(hidden_dim + self.dir_encoding.output_dim, hidden_dim // 2)
        self.rgb_head = nn.Linear(hidden_dim // 2, 3)

    def forward(self, points, directions):
        if points.shape != directions.shape or points.shape[-1] != 3:
            raise ValueError("points and directions must align with last dimension 3")
        if not torch.isfinite(points).all() or not torch.isfinite(directions).all():
            raise ValueError("points/directions must be finite")
        norms = directions.norm(dim=-1)
        if not torch.allclose(norms, torch.ones_like(norms), atol=1e-4, rtol=1e-4):
            raise ValueError("view directions must be unit vectors")
        xyz = self.xyz_encoding(points)
        hidden = xyz
        for index, layer in enumerate(self.position_layers):
            if index == self.skip_layer:
                hidden = torch.cat([hidden, xyz], dim=-1)
            hidden = F.relu(layer(hidden))
        sigma = F.softplus(self.sigma_head(hidden)).squeeze(-1)
        feature = self.feature_head(hidden)
        direction = self.dir_encoding(directions)
        color_hidden = F.relu(self.direction_layer(torch.cat([feature, direction], dim=-1)))
        rgb = torch.sigmoid(self.rgb_head(color_hidden))
        return sigma, rgb

nerf_probe48 = NeRFMLP()
points_probe48 = torch.randn(2, 7, 3, requires_grad=True)
dirs_probe48 = F.normalize(torch.randn(2, 7, 3), dim=-1)
sigma_probe48, rgb_probe48 = nerf_probe48(points_probe48, dirs_probe48)
assert sigma_probe48.shape == (2, 7) and rgb_probe48.shape == (2, 7, 3)
assert bool((sigma_probe48 >= 0).all()) and bool(((rgb_probe48 >= 0) & (rgb_probe48 <= 1)).all())
(sigma_probe48.mean() + rgb_probe48.mean()).backward()
assert points_probe48.grad is not None and torch.isfinite(points_probe48.grad).all()
assert nerf_probe48.position_layers[3].in_features == 48 + 33


## 5. Stratified sampling：随机性必须由调用方 generator 控制

把 `[near,far]` 等分为 `S` 个 bin，每个 bin 内采一个点。训练通常 `perturb=True`，推理可取 bin 中点。函数不读取隐藏的全局 RNG：随机模式必须显式传 `torch.Generator`，这样重试、测试和多 worker 场景才能复现。每条 ray 的采样深度严格递增且落在合同区间内。


In [ ]:
def stratified_samples(ray_count, near, far, sample_count, perturb, generator=None):
    if ray_count <= 0 or sample_count < 2 or not (math.isfinite(near) and math.isfinite(far) and 0 <= near < far):
        raise ValueError("invalid ray/sample/near/far contract")
    if perturb and generator is None:
        raise ValueError("perturbed sampling requires an explicit generator")
    edges = torch.linspace(near, far, sample_count + 1)
    lower = edges[:-1].expand(ray_count, -1)
    upper = edges[1:].expand(ray_count, -1)
    if perturb:
        eps = torch.finfo(torch.float32).eps
        unit = torch.rand(ray_count, sample_count, generator=generator).clamp(eps, 1 - eps)
    else:
        unit = torch.full((ray_count, sample_count), 0.5)
    return lower + (upper - lower) * unit

z_mid48 = stratified_samples(3, 0.5, 2.5, 8, False)
z_random_a48 = stratified_samples(3, 0.5, 2.5, 8, True, torch.Generator().manual_seed(9))
z_random_b48 = stratified_samples(3, 0.5, 2.5, 8, True, torch.Generator().manual_seed(9))
assert torch.equal(z_random_a48, z_random_b48)
assert not torch.equal(z_mid48, z_random_a48)
assert bool((z_random_a48[:, 1:] > z_random_a48[:, :-1]).all())
assert float(z_random_a48.min()) >= 0.5 and float(z_random_a48.max()) <= 2.5
try:
    stratified_samples(1, 2.0, 1.0, 8, False)
    raise AssertionError("near >= far must fail")
except ValueError:
    pass


## 6. Alpha compositing：用常密度场对照解析解

采样点代表 Voronoi 区间：内部边界取相邻深度中点，首尾边界固定为 `near/far`，所以所有 $\delta_i$ 之和精确等于 `far-near`。离散量为

$$\alpha_i=1-e^{-\sigma_i\delta_i},\quad
T_i=\prod_{j<i}(1-\alpha_j),\quad w_i=T_i\alpha_i.$$

常密度 $\sigma$ 时，累计不透明度应精确接近 $1-e^{-\sigma(far-near)}$。剩余透射率乘背景颜色；若把最后 delta 设为“无限大”，就无法正确表达指定 far plane 和透明背景。


In [ ]:
def volume_render(sigma, rgb, z_values, near, far, white_background=True):
    if sigma.ndim != 2 or rgb.shape != sigma.shape + (3,) or z_values.shape != sigma.shape:
        raise ValueError("expected sigma/z [R,S] and rgb [R,S,3]")
    if not (math.isfinite(near) and math.isfinite(far) and 0 <= near < far):
        raise ValueError("invalid near/far")
    if not torch.isfinite(sigma).all() or not torch.isfinite(rgb).all() or not torch.isfinite(z_values).all():
        raise ValueError("render inputs must be finite")
    if (sigma < 0).any() or (rgb < 0).any() or (rgb > 1).any():
        raise ValueError("sigma must be nonnegative and rgb in [0,1]")
    if (z_values[:, 1:] <= z_values[:, :-1]).any() or (z_values[:, 0] <= near).any() or (z_values[:, -1] >= far).any():
        raise ValueError("z samples must be strictly increasing inside (near,far)")
    internal = (z_values[:, :-1] + z_values[:, 1:]) / 2
    edges = torch.cat([torch.full_like(z_values[:, :1], near), internal,
                       torch.full_like(z_values[:, :1], far)], dim=-1)
    deltas = edges[:, 1:] - edges[:, :-1]
    alpha = 1 - torch.exp(-sigma * deltas)
    transmittance = torch.cumprod(torch.cat([torch.ones_like(alpha[:, :1]), 1 - alpha], dim=-1), dim=-1)[:, :-1]
    weights = transmittance * alpha
    remaining = torch.prod(1 - alpha, dim=-1)
    color = (weights[..., None] * rgb).sum(dim=-2)
    if white_background:
        color = color + remaining[:, None]
    depth = (weights * z_values).sum(dim=-1)
    return {"rgb": color, "depth": depth, "weights": weights, "remaining": remaining,
            "alpha": alpha, "deltas": deltas}

constant_z48 = stratified_samples(2, 1.0, 3.0, 16, False)
constant_sigma48 = torch.full((2, 16), 2.0)
constant_rgb48 = torch.full((2, 16, 3), 0.25)
constant_render48 = volume_render(constant_sigma48, constant_rgb48, constant_z48, 1.0, 3.0, True)
expected_opacity48 = 1 - math.exp(-2.0 * (3.0 - 1.0))
expected_color48 = 0.25 * expected_opacity48 + (1 - expected_opacity48)
assert torch.allclose(constant_render48["weights"].sum(-1), torch.full((2,), expected_opacity48), atol=1e-6)
assert torch.allclose(constant_render48["rgb"], torch.full((2, 3), expected_color48), atol=1e-6)
assert torch.allclose(constant_render48["weights"].sum(-1) + constant_render48["remaining"], torch.ones(2), atol=1e-6)
empty_render48 = volume_render(torch.zeros(1, 8), torch.zeros(1, 8, 3),
                               stratified_samples(1, 0.1, 1.1, 8, False), 0.1, 1.1, True)
assert torch.equal(empty_render48["rgb"], torch.ones(1, 3))
assert torch.equal(empty_render48["weights"], torch.zeros(1, 8))


## 7. Hierarchical sampling：从 coarse weights 构造分段常数 PDF

coarse 渲染的每个 sample 对应一个 depth bin。给 bin edges 和非负 weights，先归一化为 PDF、累加成 CDF，再用 inverse transform sampling 抽取 fine depths。训练可用随机 `u`，确定性推理用等距分位点。采样使用 `searchsorted`，边界 index 必须 clamp，零权重时以 epsilon 退化为近似均匀分布。


In [ ]:
def sample_pdf(bin_edges, weights, sample_count, deterministic, generator=None):
    if bin_edges.ndim != 2 or weights.ndim != 2 or bin_edges.shape[0] != weights.shape[0] or bin_edges.shape[1] != weights.shape[1] + 1:
        raise ValueError("bin_edges [R,M+1] and weights [R,M] required")
    if (sample_count <= 0 or not torch.is_floating_point(bin_edges) or not torch.is_floating_point(weights)
            or not torch.isfinite(bin_edges).all() or not torch.isfinite(weights).all() or (weights < 0).any()):
        raise ValueError("invalid PDF inputs")
    if bin_edges.dtype != weights.dtype or bin_edges.device != weights.device:
        raise ValueError("PDF edges/weights must share dtype and device")
    if (bin_edges[:, 1:] <= bin_edges[:, :-1]).any():
        raise ValueError("bin edges must be strictly increasing")
    if not deterministic and generator is None:
        raise ValueError("random PDF sampling requires an explicit generator")
    stabilized = weights + 1e-5
    pdf = stabilized / stabilized.sum(-1, keepdim=True)
    cdf = torch.cat([torch.zeros_like(pdf[:, :1]), torch.cumsum(pdf, -1)], -1)
    if deterministic:
        u = (torch.arange(sample_count, dtype=weights.dtype, device=weights.device) + 0.5) / sample_count
        u = u.expand(weights.shape[0], -1).contiguous()
    else:
        eps = torch.finfo(weights.dtype).eps
        u = torch.rand(weights.shape[0], sample_count, generator=generator,
                       dtype=weights.dtype, device=weights.device).clamp(eps, 1 - eps)
    indices = torch.searchsorted(cdf.contiguous(), u.contiguous(), right=True)
    below = (indices - 1).clamp(0, cdf.shape[1] - 2)
    above = (below + 1).clamp(max=cdf.shape[1] - 1)
    cdf_low = torch.gather(cdf, 1, below); cdf_high = torch.gather(cdf, 1, above)
    edge_low = torch.gather(bin_edges, 1, below); edge_high = torch.gather(bin_edges, 1, above)
    fraction = (u - cdf_low) / (cdf_high - cdf_low).clamp(min=1e-8)
    return edge_low + fraction * (edge_high - edge_low)

pdf_edges48 = torch.tensor([[0., 1., 2., 3., 4.]])
pdf_weights48 = torch.tensor([[0., 0., 10., 0.]])
focused_samples48 = sample_pdf(pdf_edges48, pdf_weights48, 8, True)
assert focused_samples48.shape == (1, 8)
assert bool(((focused_samples48 > 2.0) & (focused_samples48 < 3.0)).all())
random_pdf_a48 = sample_pdf(pdf_edges48, torch.ones_like(pdf_weights48), 6, False, torch.Generator().manual_seed(7))
random_pdf_b48 = sample_pdf(pdf_edges48, torch.ones_like(pdf_weights48), 6, False, torch.Generator().manual_seed(7))
assert torch.equal(random_pdf_a48, random_pdf_b48)


## 8. Coarse/fine renderer：固定数量、严格排序与重复深度语义

coarse pass 得到 weights 后，将其 `detach` 构造 PDF；fine samples 与 coarse depths 合并排序，再重新查询同一 MLP。本教学版共享 coarse/fine 网络以减少参数，原论文使用两套网络。

透明场、均匀 PDF 且 `fine_samples==coarse_samples` 时，确定性 fine 分位点会与 coarse 中点**必然重合**。这不是随机重试能解决的问题。本实现保留 `coarse+fine` 个 query：排序后若相等，把后一个值用 `torch.nextafter(previous,+∞)` 向前推进一个 ULP；随后重新验证 `near < z < far` 和严格递增。这个极小数值扰动被明确写入 render recipe。


In [ ]:
def make_strict_depths48(sorted_depths, near, far):
    if sorted_depths.ndim != 2 or sorted_depths.shape[1] < 2 or not torch.is_floating_point(sorted_depths):
        raise ValueError("sorted depths must be floating [R,S>=2]")
    if not torch.isfinite(sorted_depths).all() or bool((sorted_depths[:, 1:] < sorted_depths[:, :-1]).any()):
        raise ValueError("depths must be finite and nondecreasing before collision repair")
    columns = [sorted_depths[:, 0]]
    toward = torch.full_like(columns[0], float("inf"))
    for index in range(1, sorted_depths.shape[1]):
        current = sorted_depths[:, index]
        bumped = torch.nextafter(columns[-1], toward)
        columns.append(torch.where(current <= columns[-1], bumped, current))
    strict = torch.stack(columns, dim=1)
    if (strict[:, 0] <= near).any() or (strict[:, -1] >= far).any() or (strict[:, 1:] <= strict[:, :-1]).any():
        raise RuntimeError("floating precision cannot represent distinct hierarchical depths inside bounds")
    return strict

class CoarseFineRenderer(nn.Module):
    def __init__(self, field, near=0.7, far=3.5, coarse_samples=12, fine_samples=8, white_background=True):
        super().__init__()
        if not (math.isfinite(near) and math.isfinite(far) and 0 <= near < far) or coarse_samples < 2 or fine_samples <= 0:
            raise ValueError("invalid renderer configuration")
        self.field = field
        self.near, self.far = float(near), float(far)
        self.coarse_samples, self.fine_samples = int(coarse_samples), int(fine_samples)
        self.white_background = bool(white_background)

    def _query(self, origins, directions, z_values):
        points = origins[:, None, :] + directions[:, None, :] * z_values[..., None]
        view_dirs = directions[:, None, :].expand_as(points)
        sigma, rgb = self.field(points, view_dirs)
        return sigma, rgb

    def forward(self, origins, directions, perturb=False, generator=None):
        if origins.ndim != 2 or origins.shape != directions.shape or origins.shape[-1] != 3 or origins.shape[0] == 0:
            raise ValueError("origins/directions must be nonempty [R,3]")
        if (not torch.is_floating_point(origins) or not torch.is_floating_point(directions)
                or origins.dtype != directions.dtype or origins.device != directions.device):
            raise ValueError("origins/directions must share floating dtype/device")
        if origins.dtype != torch.float32:
            raise ValueError("published renderer expects float32 rays")
        if not torch.isfinite(origins).all() or not torch.isfinite(directions).all():
            raise ValueError("ray origins/directions must be finite")
        if not torch.allclose(directions.norm(dim=-1), torch.ones(directions.shape[0]), atol=1e-4):
            raise ValueError("renderer directions must be normalized")
        z_coarse = stratified_samples(origins.shape[0], self.near, self.far, self.coarse_samples, perturb, generator)
        sigma_c, rgb_c = self._query(origins, directions, z_coarse)
        coarse = volume_render(sigma_c, rgb_c, z_coarse, self.near, self.far, self.white_background)
        internal = (z_coarse[:, :-1] + z_coarse[:, 1:]) / 2
        edges = torch.cat([torch.full_like(z_coarse[:, :1], self.near), internal,
                           torch.full_like(z_coarse[:, :1], self.far)], -1)
        z_fine = sample_pdf(edges, coarse["weights"].detach(), self.fine_samples,
                            deterministic=not perturb, generator=generator)
        z_sorted = torch.sort(torch.cat([z_coarse, z_fine], -1), dim=-1).values
        z_all = make_strict_depths48(z_sorted, self.near, self.far)
        sigma_f, rgb_f = self._query(origins, directions, z_all)
        fine = volume_render(sigma_f, rgb_f, z_all, self.near, self.far, self.white_background)
        return {"coarse": coarse, "fine": fine, "z_coarse": z_coarse, "z_fine": z_fine, "z_all": z_all}

renderer_probe48 = CoarseFineRenderer(nerf_probe48)
probe_origins48 = torch.zeros(4, 3)
probe_dirs_render48 = F.normalize(torch.tensor([[0., 0., -1.], [.1, 0., -1.], [0., .1, -1.], [.1, .1, -1.]]), dim=-1)
render_probe_a48 = renderer_probe48(probe_origins48, probe_dirs_render48, True, torch.Generator().manual_seed(12))
render_probe_b48 = renderer_probe48(probe_origins48, probe_dirs_render48, True, torch.Generator().manual_seed(12))
assert torch.equal(render_probe_a48["z_all"], render_probe_b48["z_all"])
assert torch.equal(render_probe_a48["fine"]["rgb"], render_probe_b48["fine"]["rgb"])
assert render_probe_a48["fine"]["rgb"].shape == (4, 3)
assert render_probe_a48["z_all"].shape == (4, 20)

class ZeroDensityField48(nn.Module):
    def forward(self, points, directions):
        return points.new_zeros(points.shape[:-1]), points.new_zeros(points.shape[:-1] + (3,))

collision_renderer48 = CoarseFineRenderer(
    ZeroDensityField48(), near=0.0, far=1.0, coarse_samples=4, fine_samples=4, white_background=True)
collision_render48 = collision_renderer48(
    torch.zeros(1, 3), torch.tensor([[0., 0., -1.]]), perturb=False)
assert torch.equal(collision_render48["z_coarse"], collision_render48["z_fine"])
assert collision_render48["z_all"].shape == (1, 8)
assert bool((collision_render48["z_all"][:, 1:] > collision_render48["z_all"][:, :-1]).all())
assert float(collision_render48["z_all"].min()) > 0 and float(collision_render48["z_all"].max()) < 1
assert torch.equal(collision_render48["fine"]["rgb"], torch.ones(1, 3))

bad_rays48 = [
    (torch.tensor([[float("nan"), 0., 0.]]), torch.tensor([[0., 0., -1.]])),
    (torch.empty(0, 3), torch.empty(0, 3)),
    (torch.zeros(1, 3, dtype=torch.float64), torch.tensor([[0., 0., -1.]], dtype=torch.float32)),
]
for bad_origins48, bad_directions48 in bad_rays48:
    try:
        collision_renderer48(bad_origins48, bad_directions48, perturb=False)
        raise AssertionError("invalid ray contract must fail")
    except ValueError:
        pass


## 9. 解析微型辐射场与受控拟合

目标场是原点附近的平滑“雾球”：density 为高斯，颜色由位置和少量视角项决定。训练点均匀采自 `[-1,1]^3`，train/validation 用固定、互不重叠的索引。直接监督 `(sigma,rgb)` 比只用图像重建更容易，是为了快速验证 MLP、skip 和渲染梯度；它是**乐观的受控单元测试**，不等同于从多视图图片恢复几何。


In [ ]:
def analytic_field48(points, directions):
    radius2 = points.square().sum(-1)
    sigma = 7.0 * torch.exp(-5.0 * radius2)
    rgb = torch.sigmoid(1.8 * points + 0.25 * directions)
    return sigma, rgb

field_generator48 = torch.Generator().manual_seed(SEED + 1)
all_points48 = torch.rand(768, 3, generator=field_generator48) * 2 - 1
all_directions48 = F.normalize(torch.randn(768, 3, generator=field_generator48), dim=-1)
all_sigma48, all_rgb48 = analytic_field48(all_points48, all_directions48)
train_indices48 = torch.arange(640)
validation_indices48 = torch.arange(640, 768)
assert set(train_indices48.tolist()).isdisjoint(validation_indices48.tolist())

nerf_model48 = NeRFMLP(hidden_dim=48, depth=5, skip_layer=3)
optimizer48 = torch.optim.Adam(nerf_model48.parameters(), lr=4e-3)
loss_trace48 = []
for step in range(140):
    batch_generator = torch.Generator().manual_seed(SEED + 100 + step)
    choice = train_indices48[torch.randperm(train_indices48.numel(), generator=batch_generator)[:192]]
    pred_sigma, pred_rgb = nerf_model48(all_points48[choice], all_directions48[choice])
    loss = F.mse_loss(pred_sigma / 7.0, all_sigma48[choice] / 7.0) + F.mse_loss(pred_rgb, all_rgb48[choice])
    optimizer48.zero_grad(set_to_none=True); loss.backward(); optimizer48.step()
    loss_trace48.append(float(loss.detach()))

nerf_model48.eval()
with torch.no_grad():
    val_sigma48, val_rgb48 = nerf_model48(all_points48[validation_indices48], all_directions48[validation_indices48])
    validation_loss48 = float(F.mse_loss(val_sigma48 / 7.0, all_sigma48[validation_indices48] / 7.0)
                              + F.mse_loss(val_rgb48, all_rgb48[validation_indices48]))
assert loss_trace48[-1] < 0.3 * loss_trace48[0]
assert validation_loss48 < 0.035
assert all(math.isfinite(v) for v in loss_trace48)
print({"initial_field_loss": round(loss_trace48[0], 5), "final_field_loss": round(loss_trace48[-1], 5),
       "validation_field_loss": round(validation_loss48, 5)})


## 10. Novel-view 检查：只在声明的视角包络内解释结果

`look_at_pose` 构造 camera-to-world：rotation 三列分别是 right、up、back，因此 camera 的 `-z` 指向目标。我们声明训练/验证用途的允许方位角为 `[-25°,25°]`，在 `15°` 渲染一次；`60°` 必须被 evaluator 拒绝，而不是把外插结果包装成 novel-view 成功。

由于前一节直接监督了三维场，这里的图像误差只检查 ray/采样/compositing 是否与解析场一致。


In [ ]:
def validate_view_angle48(angle_degrees, allowed=(-25.0, 25.0)):
    if not math.isfinite(angle_degrees) or not allowed[0] <= angle_degrees <= allowed[1]:
        raise ValueError("view is outside the declared evaluation envelope")

validate_view_angle48(15.0)
try:
    validate_view_angle48(60.0)
    raise AssertionError("out-of-envelope view must not be reported")
except ValueError:
    pass

camera48 = PinholeRayGenerator(8, 8, focal=10.0)
novel_pose48 = look_at_pose48(15.0, 2.0).unsqueeze(0)
assert camera_matrix_digest48(novel_pose48) == _CAMERA_REGISTRY48["eval-azimuth-15-radius-2"]
novel_camera48 = trusted_camera_to_world48("eval-azimuth-15-radius-2", novel_pose48)
novel_origins48, novel_dirs48 = camera48(novel_camera48)
renderer48 = CoarseFineRenderer(nerf_model48, near=0.7, far=3.5, coarse_samples=14, fine_samples=10)
with torch.no_grad():
    predicted_view48 = renderer48(novel_origins48, novel_dirs48, perturb=False)["fine"]["rgb"]
    dense_z48 = stratified_samples(novel_origins48.shape[0], 0.7, 3.5, 48, False)
    dense_points48 = novel_origins48[:, None] + novel_dirs48[:, None] * dense_z48[..., None]
    dense_dirs48 = novel_dirs48[:, None].expand_as(dense_points48)
    true_sigma_view48, true_rgb_view48 = analytic_field48(dense_points48, dense_dirs48)
    true_view48 = volume_render(true_sigma_view48, true_rgb_view48, dense_z48, 0.7, 3.5, True)["rgb"]
    novel_view_mse48 = float(F.mse_loss(predicted_view48, true_view48))
assert predicted_view48.shape == (64, 3)
assert torch.isfinite(predicted_view48).all()
assert novel_view_mse48 < 0.06
assert torch.allclose(novel_dirs48.norm(dim=-1), torch.ones(64), atol=1e-6)
print({"in-envelope_angle": 15.0, "analytic_render_mse": round(novel_view_mse48, 6)})


## 11. 发布制品：field、相机、编码顺序和 renderer 必须作为一个系统交付

NeRF 权重不能脱离 camera convention、受信 pose、focal、near/far、位置编码的 $\pi$/排列顺序、方向归一化和背景解释。package 内部摘要只能发现传输损坏，不能认证发布者，因此仍由 package 外 publisher registry 固定整体 digest。

loader 逐项交叉验证 config、encoding、camera registry、render/preprocess、训练 split 和实际 evaluation pose，再返回不可变 `PublishedNeRF` bundle。bundle 能从 `CameraToWorld` 生成 rays 并构造固定 renderer；调用方不再拿裸 MLP 猜坐标。伪造测试替换 RGB head 并重算所有内部 hash，仍由外部信任锚拒绝。


In [ ]:
def state_digest48(state):
    digest = sha256()
    for key in sorted(state):
        tensor = state[key].detach().cpu().contiguous()
        digest.update(key.encode()); digest.update(str(tensor.dtype).encode())
        digest.update(json.dumps(list(tensor.shape)).encode()); digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()

def tensor_digest48(*tensors):
    digest = sha256()
    for tensor in tensors:
        value = tensor.detach().cpu().contiguous()
        digest.update(str(value.dtype).encode()); digest.update(json.dumps(list(value.shape)).encode())
        digest.update(value.numpy().tobytes())
    return digest.hexdigest()

def package_digest48(package):
    payload = {k: package[k] for k in sorted(package) if k != "package_digest"}
    return sha256(json.dumps(payload, sort_keys=True, separators=(",", ":")).encode()).hexdigest()

def deep_freeze48(value):
    if isinstance(value, dict): return MappingProxyType({k: deep_freeze48(v) for k, v in value.items()})
    if isinstance(value, list): return tuple(deep_freeze48(v) for v in value)
    return value

CONFIG48 = {"xyz_frequencies": 5, "dir_frequencies": 2, "include_input": True,
            "hidden_dim": 48, "depth": 5, "skip_layer": 3}
ENCODING_RECIPE48 = {"frequency": "2**k*pi", "k": "0..L-1",
                     "feature_order": "input,all-sin-input-major-frequency-minor,all-cos-input-major-frequency-minor",
                     "xyz_dim": 33, "direction_dim": 15}
SPLIT48 = {"field_seed": SEED + 1, "train": [0, 640], "validation": [640, 768],
           "selection": "fixed-steps-validation-report-only"}
PREPROCESS48 = {"points": "finite-world-xyz-float32", "directions": "finite-unit-world-float32",
                "rgb_range": [0.0, 1.0], "pose_input": "trusted-CameraToWorld"}
CAMERA_RECIPE48 = {"matrix": "camera-to-world", "structural_check": "proper-rigid-not-direction-proof",
                   "forward": "camera-negative-z", "pixel": "center", "image_index_y": "down",
                   "camera_axis_y": "up", "direction": "unit-world",
                   "registered_matrix_digests": dict(_CAMERA_REGISTRY48)}
RENDER_RECIPE48 = {"near": 0.7, "far": 3.5, "coarse": 14, "fine": 10,
                   "background": "white", "depth_bins": "midpoint-voronoi",
                   "coarse_weight_to_pdf": "detach", "duplicate_depth": "nextafter-positive-one-ulp-keep-count"}
EVALUATION_RECIPE48 = {"image_size": [8, 8], "focal": 10.0,
                       "camera_id": "eval-azimuth-15-radius-2",
                       "camera_matrix_digest": _CAMERA_REGISTRY48["eval-azimuth-15-radius-2"],
                       "azimuth_degrees": 15.0, "radius": 2.0,
                       "allowed_azimuth_degrees": [-25.0, 25.0], "dense_reference_samples": 48}
TRAINING_RECIPE48 = {"seed": SEED, "optimizer": "Adam", "lr": 4e-3, "steps": 140,
                     "batch_size": 192, "objective": "direct-field-MSE", "controlled_fixture": True}

state48 = {k: v.detach().cpu().clone() for k, v in nerf_model48.state_dict().items()}
buffer48 = io.BytesIO(); torch.save(state48, buffer48)
artifact48 = {
    "artifact_id": "analytic-mini-nerf-v1", "config": CONFIG48,
    "state_hex": buffer48.getvalue().hex(), "state_digest": state_digest48(state48),
    "data_digest": tensor_digest48(all_points48, all_directions48, all_sigma48, all_rgb48,
                                   train_indices48, validation_indices48),
    "split": SPLIT48, "encoding_recipe": ENCODING_RECIPE48, "preprocess": PREPROCESS48,
    "camera_recipe": CAMERA_RECIPE48, "render_recipe": RENDER_RECIPE48,
    "evaluation": EVALUATION_RECIPE48, "training_recipe": TRAINING_RECIPE48,
}
artifact48["package_digest"] = package_digest48(artifact48)
PUBLISHER_REGISTRY48 = MappingProxyType({artifact48["artifact_id"]: artifact48["package_digest"]})

@dataclass(frozen=True)
class PublishedNeRF:
    field: NeRFMLP
    config: object
    encoding: object
    preprocess: object
    camera: object
    render_recipe: object
    evaluation: object
    split: object

    def rays(self, camera_to_world):
        height, width = self.evaluation["image_size"]
        generator = PinholeRayGenerator(height, width, float(self.evaluation["focal"]))
        return generator(camera_to_world)

    def renderer(self):
        return CoarseFineRenderer(self.field, float(self.render_recipe["near"]),
                                  float(self.render_recipe["far"]), int(self.render_recipe["coarse"]),
                                  int(self.render_recipe["fine"]), self.render_recipe["background"] == "white")

    def render_camera(self, camera_to_world, perturb=False, generator=None):
        origins, directions = self.rays(camera_to_world)
        return self.renderer()(origins, directions, perturb=perturb, generator=generator)

def load_published_nerf48(package):
    artifact_id = package.get("artifact_id")
    if artifact_id not in PUBLISHER_REGISTRY48 or package.get("package_digest") != PUBLISHER_REGISTRY48[artifact_id]:
        raise ValueError("artifact is not approved by publisher registry")
    if package_digest48(package) != package["package_digest"]:
        raise ValueError("package digest mismatch")
    expected = {"config": CONFIG48, "split": SPLIT48, "encoding_recipe": ENCODING_RECIPE48,
                "preprocess": PREPROCESS48, "camera_recipe": CAMERA_RECIPE48,
                "render_recipe": RENDER_RECIPE48, "evaluation": EVALUATION_RECIPE48,
                "training_recipe": TRAINING_RECIPE48}
    for field, wanted in expected.items():
        if package.get(field) != wanted:
            raise ValueError(f"published NeRF contract mismatch: {field}")
    expected_data = tensor_digest48(all_points48, all_directions48, all_sigma48, all_rgb48,
                                    train_indices48, validation_indices48)
    if package.get("data_digest") != expected_data:
        raise ValueError("bound field split mismatch")
    state = torch.load(io.BytesIO(bytes.fromhex(package["state_hex"])), map_location="cpu", weights_only=True)
    if state_digest48(state) != package["state_digest"]:
        raise ValueError("canonical state digest mismatch")
    cfg = package["config"]
    model = NeRFMLP(cfg["xyz_frequencies"], cfg["dir_frequencies"], cfg["hidden_dim"],
                    cfg["depth"], cfg["skip_layer"]).eval()
    model.load_state_dict(state, strict=True)
    return PublishedNeRF(model, deep_freeze48(cfg), deep_freeze48(package["encoding_recipe"]),
                         deep_freeze48(package["preprocess"]), deep_freeze48(package["camera_recipe"]),
                         deep_freeze48(package["render_recipe"]), deep_freeze48(package["evaluation"]),
                         deep_freeze48(package["split"]))

loaded_nerf48 = load_published_nerf48(deepcopy(artifact48))
with torch.no_grad():
    loaded_sigma48, loaded_rgb48 = loaded_nerf48.field(all_points48[:4], all_directions48[:4])
    original_sigma48, original_rgb48 = nerf_model48(all_points48[:4], all_directions48[:4])
assert torch.equal(loaded_sigma48, original_sigma48) and torch.equal(loaded_rgb48, original_rgb48)
loaded_origins48, loaded_dirs48 = loaded_nerf48.rays(novel_camera48)
assert torch.equal(loaded_origins48, novel_origins48) and torch.equal(loaded_dirs48, novel_dirs48)
assert loaded_nerf48.camera["structural_check"] == "proper-rigid-not-direction-proof"
try:
    loaded_nerf48.evaluation["focal"] = 9.0
    raise AssertionError("published evaluation recipe should be read-only")
except TypeError:
    pass

forged48 = deepcopy(artifact48)
forged_state48 = torch.load(io.BytesIO(bytes.fromhex(forged48["state_hex"])), weights_only=True)
forged_state48["rgb_head.bias"] = forged_state48["rgb_head.bias"] + 4
forged_buffer48 = io.BytesIO(); torch.save(forged_state48, forged_buffer48)
forged48["state_hex"] = forged_buffer48.getvalue().hex()
forged48["state_digest"] = state_digest48(forged_state48)
forged48["evaluation"]["focal"] = 9.0
forged48["package_digest"] = package_digest48(forged48)
try:
    load_published_nerf48(forged48)
    raise AssertionError("self-rehashed field replacement must fail closed")
except ValueError as exc:
    assert "publisher registry" in str(exc)


## 12. 失败模式、复杂度与生产差距

- **near/far**：颠倒、相等、sample 落在端点外都会拒绝；真实场景需由相机/scene bounds 稳健估计。
- **随机性**：stratified/PDF sampling 只接受显式 generator；分布式训练还需按 rank/step 派生互不重叠的随机流。
- **数值稳定**：大 sigma 下 `1-alpha` 可能下溢为 0，这是完全不透明的合理极限；混合精度需监控 NaN/Inf 和 transmittance。
- **复杂度**：MLP 查询量约为 `rays × (coarse+fine)`；本实现逐像素全量推理，生产需 ray batching、occupancy grid/空空间跳跃与 fused kernels。
- **相机语义**：刚体结构检查无法区分 C2W/W2C；`CameraToWorld` 必须由相机 ID 与矩阵 digest 的包外 registry 认证，并保留已知平移/中心 ray oracle。
- **发布语义**：`PublishedNeRF` 将 field、encoding、camera、preprocess、renderer 与真实 evaluation pose 作为不可变 bundle 交付。
- **质量边界**：这里直接监督解析三维场，15° 也只在声明包络内。真实 NeRF 要用多视图 photometric loss、holdout cameras、PSNR/SSIM/LPIPS、pose 误差与曝光变化分析。

原始资料：

- [NeRF: Representing Scenes as Neural Radiance Fields for View Synthesis](https://arxiv.org/abs/2003.08934)
- [Fourier Features Let Networks Learn High Frequency Functions](https://arxiv.org/abs/2006.10739)
- [PyTorch searchsorted 文档](https://pytorch.org/docs/stable/generated/torch.searchsorted.html)
